# Check the results of the Rubin metadetect shear matching

In [ ]:
# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import tables_io
import hpmcm

In [ ]:
# set the tract
tract = 8006

In [ ]:
# Input shear catalog files and matching output tables
shear_rubin_in = tables_io.read(f'{tract}/shear_object_{tract}.parq')
shear_rubin_ns = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_ns.parq')
shear_rubin_1m = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_1m.parq')
shear_rubin_1p = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_1p.parq')
shear_rubin_2m = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_2m.parq')
shear_rubin_2p = tables_io.read(f'{tract}/shear_object_reduced_uncleaned_{tract}_2p.parq')    
shear_rubin_obj_stats = tables_io.read(f'{tract}/rubin_md_match_object_stats.pq')
shear_rubin_obj_assoc = tables_io.read(f'{tract}/rubin_md_match_object_assoc.pq')
input_files = [
    f'{tract}/shear_object_reduced_uncleaned_{tract}_ns.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_1p.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2m.parq',
    f'{tract}/shear_object_reduced_uncleaned_{tract}_2p.parq',
]    
catalog_ids = [0, 1, 2, 3, 4]

In [ ]:
(shear_rubin_obj_stats.n_sib_objects == 2).sum()

In [ ]:
np.bincount(shear_rubin_obj_stats.n_sib_srcs)

## Validation plots

In [ ]:
_ = plt.scatter(shear_rubin_obj_stats.x_pix, shear_rubin_obj_stats.y_pix, s=1)

In [ ]:
_ = plt.scatter(shear_rubin_ns.x_pix, shear_rubin_ns.y_pix, s=1)

In [ ]:
joined_shear_obj_full = hpmcm.output_tables.buildJoinedObjectTable(
    shear_rubin_obj_stats,
    shear_rubin_obj_assoc,
    input_files,
    catalog_ids,
)

In [ ]:
(joined_shear_obj_full.n_src > joined_shear_obj_full.n_unique).sum()

In [ ]:
center_mask = np.bitwise_and(np.fabs(joined_shear_obj_full.x_cent - 75) < 75, np.fabs(joined_shear_obj_full.y_cent - 75) < 75)


In [ ]:
joined_shear_obj = joined_shear_obj_full[center_mask]
joined_shear_obj_edge = joined_shear_obj_full[~center_mask]

In [ ]:
_ = plt.hist(joined_shear_obj.catalog_mask, bins=np.linspace(0.5, 31.5, 32))
_ = plt.hist(joined_shear_obj_edge.catalog_mask, bins=np.linspace(0.5, 31.5, 32))
_ = plt.yscale('log')

In [ ]:
_ = plt.scatter(joined_shear_obj.y_cell_coadd_0, 3600*(joined_shear_obj.ra - joined_shear_obj.ra_0), s=1)

In [ ]:
_ = plt.scatter(joined_shear_obj.x_cell_coadd_0, joined_shear_obj.x_pix - joined_shear_obj.x_pix_1, s=1)
_ = plt.scatter(joined_shear_obj.x_cell_coadd_0, joined_shear_obj.x_pix - joined_shear_obj.x_pix_2, s=1)

In [ ]:
_ = plt.scatter(joined_shear_obj.x_cell_coadd_0, 3600*(joined_shear_obj.ra - joined_shear_obj.ra_1), s=1)
_ = plt.scatter(joined_shear_obj.x_cell_coadd_0, 3600*(joined_shear_obj.ra - joined_shear_obj.ra_2), s=1)

In [ ]:
cut = np.bitwise_and(joined_shear_obj.n_sib_objects==1, joined_shear_obj.n_sib_srcs>5)
print(cut.sum())

In [ ]:
print('good', np.bitwise_and(joined_shear_obj.n_src==5, joined_shear_obj.n_unique==5).sum())
print('conf', (joined_shear_obj.n_src != joined_shear_obj.n_unique).sum())
print('missing 1', np.bitwise_and(joined_shear_obj.n_src==4, joined_shear_obj.n_unique==4).sum())
print('missing 2', np.bitwise_and(joined_shear_obj.n_src==3, joined_shear_obj.n_unique==3).sum())
print('missing 3', np.bitwise_and(joined_shear_obj.n_src==2, joined_shear_obj.n_unique==2).sum())
print('missing 4', np.bitwise_and(joined_shear_obj.n_src==1, joined_shear_obj.n_unique==1).sum())
print('split', np.bitwise_and(joined_shear_obj.n_sib_objects>1, joined_shear_obj.n_sib_srcs==5).sum())
print('weird', np.bitwise_and(joined_shear_obj.n_sib_objects>1, joined_shear_obj.n_sib_srcs>5).sum())
print('weird', np.bitwise_and(joined_shear_obj.n_sib_objects>1, joined_shear_obj.n_sib_srcs<5).sum())


In [ ]:
good_mask = joined_shear_obj.catalog_mask == 31
bad_mask = joined_shear_obj.catalog_mask != 31
good_sub = joined_shear_obj[good_mask]
bad_sub = joined_shear_obj[bad_mask]

missing_1_mask = np.bitwise_and(joined_shear_obj.n_src==4, joined_shear_obj.n_unique==4)
missing_2_mask = np.bitwise_and(joined_shear_obj.n_src==3, joined_shear_obj.n_unique==3)
missing_3_mask = np.bitwise_and(joined_shear_obj.n_src==2, joined_shear_obj.n_unique==2)
missing_4_mask = np.bitwise_and(joined_shear_obj.n_src==1, joined_shear_obj.n_unique==1)

missing_1 = joined_shear_obj[missing_1_mask]
missing_2 = joined_shear_obj[missing_2_mask]
missing_3 = joined_shear_obj[missing_3_mask]
missing_4 = joined_shear_obj[missing_4_mask]

In [ ]:
_ = plt.scatter(good_sub.ra, good_sub.dec, s=1)
_ = plt.scatter(bad_sub.ra, bad_sub.dec, s=1)

In [ ]:
#_ = plt.scatter(good_sub.x_cent[::5], good_sub.y_cent[::5], s=1)
#_ = plt.scatter(bad_sub.x_cent[::5], bad_sub.y_cent[::5], s=1)
#_ = plt.scatter(missing_1.x_cent, missing_1.y_cent, s=1)
_ = plt.scatter(missing_2.x_cent, missing_2.y_cent, s=1)
_ = plt.scatter(missing_3.x_cent, missing_3.y_cent, s=1)
#_ = plt.scatter(missing_4.x_cent, missing_4.y_cent, s=1)

In [ ]:
missing_2_217 = missing_2[missing_2.cell_idx == 217]
missing_3_217 = missing_3[missing_3.cell_idx == 217]

In [ ]:
joined_shear_obj[joined_shear_obj.cell_idx == 217]

In [ ]:
_ = plt.scatter(joined_shear_obj.cell_idx_x_0, joined_shear_obj.cell_idx_y_0, s=1)

In [ ]:
(joined_shear_obj.cell_idx - (201*joined_shear_obj.cell_idx_x_0 + joined_shear_obj.cell_idx_y_0)).max()

In [ ]:
((joined_shear_obj.cell_idx // 201 ) -1) % 20

In [ ]:
missing_2_217

In [ ]:
missing_3_217

In [ ]:
_ = plt.scatter(missing_1.ra, missing_1.dec, s=1)
_ = plt.scatter(missing_4.ra, missing_4.dec, s=1)

In [ ]:
_ = plt.hist2d(3600*(good_sub.ra-good_sub.ra_1), 3600*(good_sub.dec-good_sub.dec_1),
               bins=(np.linspace(-0.4, 0.4, 81), np.linspace(-0.4, 0.4, 81)), norm='log')

In [ ]:
_ = plt.hist((bad_sub.x_cent-75)**2 + (bad_sub.y_cent-75)**2, bins=np.linspace(-5, 2*75*75, 16))

In [ ]:
#_ = plt.hist(joined_shear_obj.x_cent, bins=np.linspace(5, 145, 15))
_ = plt.hist(missing_1.x_cent, bins=np.linspace(5, 145, 15), alpha=0.2, label='missing 1')
_ = plt.hist(missing_2.x_cent, bins=np.linspace(5, 145, 15), alpha=0.2, label='missing 2')
_ = plt.hist(missing_3.x_cent, bins=np.linspace(5, 145, 15), alpha=0.2, label='missing 3')
_ = plt.hist(missing_4.x_cent, bins=np.linspace(5, 145, 15), alpha=0.2, label='missing 4')
#_ = plt.yscale('log')
_ = plt.legend()

In [ ]:
the_hist, the_edges = np.histogram(shear_rubin_ns.x_cell_coadd, bins=np.linspace(0, 150, 16))
the_hist += np.histogram(shear_rubin_1m.x_cell_coadd, bins=np.linspace(0, 150, 16))[0]
the_hist += np.histogram(shear_rubin_1p.x_cell_coadd, bins=np.linspace(0, 150, 16))[0]
the_hist += np.histogram(shear_rubin_2m.x_cell_coadd, bins=np.linspace(0, 150, 16))[0]
the_hist += np.histogram(shear_rubin_2p.x_cell_coadd, bins=np.linspace(0, 150, 16))[0]
_ = plt.stairs(the_hist, the_edges)

In [ ]:
_ = plt.hist(good_sub.x_cent, bins=np.linspace(-5, 155, 161))
_ = plt.hist(bad_sub.x_cent, bins=np.linspace(-5, 155, 161))

In [ ]:
bad_sub_single_mask = bad_sub.n_sib_objects == 1

In [ ]:
_ = plt.hist(bad_sub.x_cent[bad_sub_single_mask], bins=np.linspace(-5, 155, 161))
_ = plt.hist(bad_sub.x_cent[~bad_sub_single_mask], bins=np.linspace(-5, 155, 161))

In [ ]:
_ = plt.hist(good_sub.y_cent, bins=np.linspace(-5, 15, 161))
_ = plt.hist(bad_sub.y_cent, bins=np.linspace(-5, 15, 161))

In [ ]:
_ = plt.hist2d(good_sub.x_pix-good_sub.x_pix_0, good_sub.y_pix-good_sub.y_pix_0,
               bins=(np.linspace(-4.0, 4.0, 81), np.linspace(-4.0, 4.0, 81)), norm='log')

In [ ]:
good_sub.x_cent-good_sub.x_cell_coadd_0